In [ ]:
import marimo as mo

# Prerequisite

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# Data preparation

## Toxic comments dataset

In [ ]:
import kagglehub

# Download latest version
toxic_comments_path = kagglehub.dataset_download("alexandersemiletov/toxic-russian-comments")

print("Path to dataset files:", toxic_comments_path)

In [ ]:
toxic_comments = (
    pl.scan_csv(
        toxic_comments_path+"/dataset.txt",
        has_header=False,
        new_columns=["raw"],
        separator="\n",
        quote_char=None,
    )
    .with_columns([
        # все лейблы в список: ["INSULT", "THREAT"]
        pl.col("raw")
          .str.extract_all(r"__label__[A-Z]+")
          .list.eval(pl.element().str.replace("__label__", ""))
          .alias("labels"),

        # текст — всё после последнего лейбла
        pl.col("raw")
          .str.replace_all(r"(__label__[A-Z]+,?\s*)+", "")
          .str.strip_chars()
          .alias("text"),
    ])
    .drop("raw")
    .collect()
)

print(toxic_comments.sample(3))

In [ ]:
toxic_comments

In [ ]:
toxic_comments['labels'].explode().value_counts()

In [ ]:
clean_toxic_comments = toxic_comments.with_columns(
    pl.col("labels").list.contains("NORMAL").not_().cast(dtype=pl.Int8).alias("toxic")
).drop("labels")
clean_toxic_comments

### Comments class distribution

In [ ]:
clean_toxic_comments["toxic"].value_counts().with_columns(
    (pl.col("count") / pl.col("count").sum() * 100)
        .round(2)
        .alias("percent")
)

### Comments size distribution

In [ ]:
comments_length_df = clean_toxic_comments.with_columns(
    pl.col("text").str.len_chars().alias("length")
)

In [ ]:
sns.histplot(data=comments_length_df, x="length")
plt.title("Reviews length distribution")
plt.grid()
plt.show()

In [ ]:
comments_length_df["length"].describe()

### Toxic vs length

In [ ]:
sns.violinplot(data=comments_length_df, x="toxic", y="length")
plt.yscale("log")
plt.grid()
plt.show()

## E-commerce reviews dataset

In [ ]:
ecommerce_df = pl.read_csv("data/ecommerce-data.csv", quote_char='"')
ecommerce_df.head()

In [ ]:
ecommerce_reviewed_df = (
    pl.read_csv("data/ecommerce-data-reviewed.csv", columns=["id", "review", "toxic"])
    .rename({"toxic": "label"})
)
ecommerce_reviewed_df

In [ ]:
joined_ecommerce_df = ecommerce_reviewed_df.join(ecommerce_df, on="id", how="inner")
joined_ecommerce_df

In [ ]:
positive_ecommerce_df = (
    ecommerce_df
    .filter(pl.col("sentiment") == "positive")
    .sample(1000)
    .with_columns(pl.lit("OK").alias("label"))
)

neautral_ecommerce_df = (
    ecommerce_df
    .filter(pl.col("sentiment") == "neautral")
    .sample(500)
    .with_columns(pl.lit("OK").alias("label"))
)


clean_ecommerce_df = pl.concat([
    ecommerce_reviewed_df.select(["review", "label"]),
    positive_ecommerce_df.select(["review", "label"]),
    neautral_ecommerce_df.select(["review", "label"]),
])

In [ ]:
clean_ecommerce_df.sample(10)

In [ ]:
clean_ecommerce_df.write_csv("clean_ecommerce_df.csv")

In [ ]:
clean_ecommerce_df.with_columns([
    pl.col("label").replace("", "OK")
]).rename({"review": "text"}).write_csv("ecommerce-data.csv")

In [ ]:
loaded_ecommerce_df = pl.read_csv("data/ecommerce-data.csv")
loaded_ecommerce_df.head()

In [ ]:
loaded_ecommerce_df["label"].value_counts()

## Spam reviews synthesis

In [ ]:
import re
import json

def parse_json_response(content: str) -> list[str]:
    # убираем markdown
    content = re.sub(r"```json|```", "", content).strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # если JSON обрезан - вытащим строки regex'ом
        matches = re.findall(r'"([^"\\]*(?:\\.[^"\\]*)*)"', content)
        return matches

In [ ]:
import os

In [ ]:
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=GITHUB_TOKEN,
)

spam_texts = []
target_spam_texts_lenght = 3000

while len(spam_texts) < target_spam_texts_lenght:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": """Сгенерируй 200 примеров спам-отзывов на русском языке для маркетплейса.

    Типы спама:
    - реклама сторонних сайтов ("купи дешевле на xxx.ru")
    - повторяющийся текст ("хороший товар хороший товар хороший товар")
    - бессмысленный набор слов
    - накрутка ("5 звезд лучший товар лучший продавец лучший магазин")
    - вместо xxx.ru должны быть реальные домены маркетплейсов

    Верни ТОЛЬКО валидный JSON массив строк без преамбулы и markdown:
    ["текст1", "текст2", ...]"""
        }],
        max_tokens=16384,
    )
    content = response.choices[0].message.content
    spam_texts.extend(parse_json_response(content))

In [ ]:
print(len(spam_texts))

In [ ]:
spam_df = pl.DataFrame({
    "text": spam_texts,
    "label": "SPAM"
})
spam_df

In [ ]:
spam_df.write_parquet("data/spam-data.parquet")

## Joining datasets

In [ ]:
selected_ecommerce = loaded_ecommerce_df.filter(
    pl.col("label").is_in(["OK", "TOXIC"])
)
selected_ecommerce

In [ ]:
selected_comments = clean_toxic_comments.sample(30_000).with_columns(
    pl.col("toxic")
    .map_elements(lambda x: "TOXIC" if x else "OK")
    .alias("label")
).drop("toxic")
selected_comments

In [ ]:
selected_spam = spam_df

In [ ]:
concated_df = pl.concat([selected_comments, selected_ecommerce, selected_spam])
concated_df["label"].value_counts()

## Droping nulls

In [ ]:
concated_df.null_count()

## Shuffling

In [ ]:
final_df = concated_df.drop_nulls().sample(fraction=1.0, shuffle=True)

In [ ]:
final_df

## Target encoding

In [ ]:
class_mapping = {"OK": 0, "TOXIC": 1, "SPAM": 2}

encoded_df = final_df.with_columns(
    pl.col("label").replace(class_mapping).cast(pl.Int8).alias("label")
)

In [ ]:
encoded_df.write_parquet("data/encoded_df.parquet")

## Class definition
- 0: OK
- 1: TOXIC
- 2: SPAM